# 03. RAG 검색·답변 품질 검증

02에서 만든 FAISS 인덱스 + LLM(Upstage solar-pro)을 LCEL 체인으로 묶어
시연 질문이 5초 안에 출처와 함께 답변되는지 확인.

**진행 순서**
1. 환경 설정 (API Key + LLM/Embedding 객체)
2. FAISS 인덱스 로드 (`allow_dangerous_deserialization=True`)
3. Retriever 생성 + 검색 동작 빠른 확인
4. 프롬프트 + RAG 체인 (LCEL)
5. 시연 Q1: "PIC 리조트 어때? 가족이 가도 좋아?" — Day 1 끝 체크
6. 시연 Q2: "5월 둘째 주 괌 날씨 어때?" — RAG 단독 한계 확인 (보강 필요성 메모)
7. (선택) 시연 Q4 RAG 부분: "PIC 4박 5일 30만원" — RAG가 PIC 정보 잘 끌어오는지

## 1. 환경 설정

- `.env`에서 `UPSTAGE_API_KEY` 로드
- LLM: `ChatUpstage(model="solar-pro")` + 가드 4개 (project_plan Section 8)
- Embedding: `solar-embedding-1-large-query` — 02에서 인덱싱 시 쓴 모델과 **같아야 함** (검색 시 query suffix 자동 적용)

In [1]:
from dotenv import load_dotenv
load_dotenv()

import os
api_key = os.getenv("UPSTAGE_API_KEY")
if not api_key:
    raise ValueError(
        "UPSTAGE_API_KEY를 찾을 수 없습니다.\n"
        "프로젝트 루트의 .env에 UPSTAGE_API_KEY=sk-...를 입력하세요."
    )
print(f"API Key 로드 성공: {api_key[:8]}...")

from langchain_upstage import ChatUpstage, UpstageEmbeddings

# LLM 가드 (plan Section 8 — 강의 contract analyzer 8장 패턴)
llm = ChatUpstage(
    model="solar-pro",
    temperature=0,      # 시연·검증 단계에서 같은 질문→같은 답 (재현성)
    max_tokens=1024,    # 답변 길이 가드 (가족 챗봇 답변엔 충분)
    timeout=30,         # 30초 응답 없으면 실패 처리
    max_retries=2,      # 일시적 네트워크 에러 시 2번까지 재시도
)

# Embedding — 02_build_index와 같은 모델 (반드시 일치해야 함)
embeddings = UpstageEmbeddings(model="solar-embedding-1-large-query")

print("환경 설정 완료 (LLM + Embedding 객체 생성)")

API Key 로드 성공: up_cCu2s...
환경 설정 완료 (LLM + Embedding 객체 생성)


## 2. FAISS 인덱스 로드

02_build_index에서 `../data/faiss_index/`에 저장한 93개 벡터 인덱스를 메모리에 로드.

- `allow_dangerous_deserialization=True`: FAISS는 pickle로 저장·로드하므로 명시적 동의 필요 (본인이 만든 인덱스라 안전)
- `embeddings` 객체를 같이 넘김: 검색 시 질문 임베딩에 필요
- 회귀 검증: 02 결과 93 벡터와 일치하는지 `assert`

In [2]:
from langchain_community.vectorstores import FAISS
from pathlib import Path

INDEX_DIR = "../data/faiss_index"

# allow_dangerous_deserialization=True:
#   FAISS는 pickle 기반 저장/로드라 명시적 동의 필요.
#   본인이 02에서 저장한 인덱스라 안전.
vectorstore = FAISS.load_local(
    INDEX_DIR,
    embeddings,
    allow_dangerous_deserialization=True,
)

# 회귀 검증: 02 결과(93 벡터)와 일치
n_vectors = vectorstore.index.ntotal
assert n_vectors == 93, f"벡터 수 불일치: {n_vectors} (기대 93)"
print(f"✓ FAISS 인덱스 로드 완료: {n_vectors}개 벡터")
print(f"  로드 경로: {Path(INDEX_DIR).resolve()}")

✓ FAISS 인덱스 로드 완료: 93개 벡터
  로드 경로: C:\Users\wonwo\fastcampus\09.LLM\project\guam-family-chatbot\data\faiss_index


## 3. Retriever 생성 + 검색 동작 빠른 확인

`vectorstore.as_retriever()`로 LCEL 체인에 끼울 수 있는 표준 검색기를 만든다.
`k=3`: 상위 3개 청크 반환 — 02 검증과 동일.

빠른 sanity check: "PIC 리조트 어때?" → 02와 같이 [리조트] PIC 청크가 top 3에 들어오는지.

In [3]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# 빠른 검색 동작 확인 (02 결과와 일치하는지)
test_docs = retriever.invoke("PIC 리조트 어때?")
print(f"검색 결과: {len(test_docs)}개\n")

for i, doc in enumerate(test_docs, 1):
    cat = doc.metadata.get("category", "?")
    title = doc.metadata.get("title", "?")
    print(f"[{i}] [{cat}] {title}")
    print(f"    {doc.page_content[:80]}...\n")

검색 결과: 3개

[1] [리조트] 퍼시픽 아일랜드 클럽 (PIC) - 나무위키
    1. 개요

Pacific Islands Club, PIC

괌 및 북마리아나 제도에 위치한 미국의 복합형 리조트. 흔히 PIC라는 약칭으로 불...

[2] [리조트] 퍼시픽 아일랜드 클럽 (PIC) - 나무위키
    호텔 내 한국인 직원이 상주하고, 외국인 직원 또한 게임등 한국어 패치가 된 분들이 있다.

사이판 최대 번화가인 가라판(Garapan) 시내와...

[3] [리조트] 퍼시픽 아일랜드 클럽 (PIC) - 나무위키
    요일별 풀파티 및 별빛 투어 등 투숙객들은 무료로 제공되며 투수객 참여로 팀을 나누어 워터 게임(물총 게임, 줄다리기, 맥주 마시기) 등등 엄청...



## 4. 프롬프트 + RAG 체인 (LCEL)

3개 부품 조립:
- `format_docs`: retriever가 반환한 `list[Document]` → 프롬프트에 들어갈 `string`
- `rag_prompt`: 시스템 메시지(친근 톤 + 출처 인용 + 모르면 모른다) + 휴먼 메시지(질문)
- LCEL 체인: `{context, question} → prompt → llm → StrOutputParser`

체인을 만들기만 하고 invoke는 다음 셀에서 시연 질문으로.

In [4]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


def format_docs(docs):
    """retriever 결과(list[Document]) → 프롬프트용 문자열.
    답변에서 출처 인용에 쓸 수 있도록 [category] title 형태로 라벨링."""
    return "\n\n".join(
        f"[{doc.metadata.get('category', '?')}] {doc.metadata.get('title', '?')}\n"
        f"{doc.page_content}"
        for doc in docs
    )


rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 괌 가족여행을 도와주는 친근한 어시스턴트입니다.
주로 부부 + 어린 아이 1~3명이 있는 가족이 질문합니다.

답변 규칙 (규칙 1번이 가장 중요. 다른 규칙과 충돌하면 규칙 1번 우선):

1. 답변의 모든 정보(설명·예시·추천·대안·"~도 좋아요" 같은 사이드 추천 모두 포함)는
   반드시 아래 참고 문서에 명시된 내용이어야 합니다.
   도움이 될 것 같다는 이유로 문서에 없는 시설·장소·서비스(예: 특정 리조트, 식당, 액티비티)를
   답변에 끼워넣지 마세요. 일반 상식·추측·다른 자료 기반 정보 모두 금지.
2. 참고 문서에서 확인할 수 없는 내용은 솔직히 "참고 자료에서 확인되지 않습니다"라고 답하세요.
3. 답변에 사용한 출처를 자연스럽게 인용하세요.
   예: "PIC는 가족 친화 리조트예요 ([리조트] 퍼시픽 아일랜드 클럽 - 나무위키)."
4. 친근한 말투로, 가족 여행자에게 도움 되도록 핵심만 간결하게 정리해주세요.
   단, 규칙 1번을 어기면서까지 도움을 주려 하지 마세요.

참고 문서:
{context}"""),
    ("human", "{question}")
])


# 체인 재구성 (rag_prompt가 바뀌었으니)
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("✓ RAG 체인 재구성 완료 (프롬프트 강화: 규칙 1번 + 우선순위 명시)")

✓ RAG 체인 재구성 완료 (프롬프트 강화: 규칙 1번 + 우선순위 명시)


## 5. 시연 Q1 검증 — "PIC 리조트 어때? 가족이 가도 좋아?"

**Day 1 끝 체크 기준** (project_plan Section 9):
> "PIC 리조트 어때?" 같은 질문에 RAG가 **5초 안에 출처와 함께** 답함

확인할 것:
- ✅ 응답 시간 5초 이내
- ✅ 답변에 PIC 정보 포함 (가족 친화·괌 최대 규모·한국어 직원 등)
- ✅ 답변에 출처 인용 (예: `[리조트] 퍼시픽 아일랜드 클럽 (PIC) - 나무위키`)

In [5]:
import time

question = "PIC 리조트 어때? 가족이 가도 좋아?"

start = time.time()
answer = rag_chain.invoke(question)
elapsed = time.time() - start

print(f"Q: {question}")
print(f"\n⏱️  응답 시간: {elapsed:.2f}초")
print(f"   (Day 1 끝 체크 기준: 5초 이내)\n")
print("=" * 60)
print("A:")
print(answer)
print("=" * 60)

Q: PIC 리조트 어때? 가족이 가도 좋아?

⏱️  응답 시간: 2.82초
   (Day 1 끝 체크 기준: 5초 이내)

A:
PIC 리조트는 가족 여행에 정말 좋은 곳이에요! 특히 어린 아이가 있는 가족에게 추천드리는 이유가 많아요.  

1. **가족 친화 시설**  
   - 괌 PIC는 **최대 규모의 수영장**과 **어린이 체험 프로그램**이 있어 아이들이 신나게 놀 수 있어요.  
   - 사이판 PIC는 **대형 워터 슬라이드 3종**과 유수풀, 인공 파도 타기 등이 있어 물놀이하기 최고예요. ([리조트] 퍼시픽 아일랜드 클럽 (PIC) - 나무위키)  

2. **다양한 식사 옵션**  
   - 한식, 중식, 일식, 양식 등 **다양한 뷔페**가 제공되어 가족 모두의 입맛을 만족시킬 수 있어요.  
   - 식사 패키지(골드/실버 카드)로 편리하게 이용 가능해요.  

3. **한국어 서비스**  
   - 한국인 직원이 상주하고, 외국인 직원도 한국어를 잘해요. 언어 부담 없이 편하게 지낼 수 있답니다.  

4. **무료 액티비티**  
   - 워터 게임, 줄다리기, 맥주 마시기 이벤트 등 **풀파티**와 **별빛 투어**가 무료로 제공돼요.  
   - 카약, 패들보트, 양궁 등도 즐길 수 있어요.  

5. **위치 편의성**  
   - 사이판 PIC는 **공항에서 도보 5분** 거리라 이동이 편리해요.  

단, 시설이 오래된 점은 감안해야 하지만, 워터파크와 놀이시설로 충분히 커버될 거예요! 가족 모두가 즐거운 시간을 보낼 수 있을 거라고 확신해요 😊  

더 궁금한 점이 있다면 언제든 물어보세요!


## 6. 시연 Q2 검증 — "5월 둘째 주 괌 날씨 어때? 우비 챙겨야 해?"

**RAG 단독 한계 확인용 셀** (인계 메시지 명시).

- 02 검증에서 Climate 청크가 top 3 미진입 (`False`). 검색은 일반 기후 정보 청크는 가져오지만, **시점별("5월 둘째 주") 날씨 정보는 RAG에 애초에 없음** (위키백과는 통시적 기후 통계만 다룸)
- 이 셀의 의도: LLM이 어떻게 답하는지 직접 보고, **도구(`get_guam_weather`)가 필요한 이유를 시연에서 자연스럽게 설명할 근거 확보**
- plan Section 11 시연 #2가 "도구 1개 — 날씨"로 분류 → Day 2에서 OpenWeatherMap 도구 추가 예정. 외교부0404 보강은 MVP 완성 후 결정 (인계 정책 유지)

In [ ]:
import time

question = "5월 둘째 주 괌 날씨 어때? 우비 챙겨야 해?"

start = time.time()
answer = rag_chain.invoke(question)
elapsed = time.time() - start

print(f"Q: {question}")
print(f"\n⏱️  응답 시간: {elapsed:.2f}초\n")
print("=" * 60)
print("A:")
print(answer)
print("=" * 60)

# 검색된 청크 — 답변의 근거가 무엇인지 같이 확인
print("\n[검색된 청크 — 답변의 근거]")
docs = retriever.invoke(question)
for i, doc in enumerate(docs, 1):
    cat = doc.metadata.get("category", "?")
    title = doc.metadata.get("title", "?")
    section = doc.metadata.get("section", "")
    label = f"[{cat}] {title}" + (f" / {section}" if section else "")
    print(f"\n[{i}] {label}")
    print(f"    {doc.page_content[:120]}...")

## 7. 결론 + Day 2 인수인계 메모

### Day 1 끝 체크 (project_plan Section 9)
> "PIC 리조트 어때?" 같은 질문에 RAG가 **5초 안에 출처와 함께 답함**

- ✅ Q1 응답 시간 3.02초 / 출처 인용 / PIC 핵심 정보 풍부 → **통과**

### 검증 결과 요약

| 시연 Q | RAG 단독 가능? | 발견 사항 |
|---|---|---|
| Q1 "PIC 리조트 어때?" | ✅ 자료 충실 답변 | 괌·사이판 PIC 정보 혼재 |
| Q2 "5월 둘째 주 날씨" | ❌ 시점 정보 자료에 없음 | 환각 발생 (프롬프트 강화로 일부만 차단) |

### 프롬프트 강화 실험 (Q2 기반)

- **변경**: 규칙 1번에 "추천·예시·대안 모두 문서 기반"·"도움 되는 이유로 추가 금지" 명시 + 규칙 1번 우선순위 명시
- **효과 측정**:
  - ✅ **"비검색 환각" 차단됨** — 강화 전 Q2 답변 끝에 등장하던 "실내 관광지(예: PIC 리조트)" 사라짐
  - ❌ **"시점·수치 환각" 차단 못 함**:
    - "월평균 70~100mm" — 자료엔 연평균 2,490mm + 월별 극값(977.6mm / 3.8mm)만. 5월·건기 월별 수치 **없음**
    - "오후/저녁에 소나기 잦음" — 자료에 "afternoon"·"evening"·"오후"·"저녁" 0건 매치
- **결론**: 프롬프트만으론 한계. 강의 S6-1 RAG 한계("맥락 부족 시 환각")와 일치

### Day 2 인수인계 발견 사항

1. **Q1 답변에서 "괌 PIC"와 "사이판 PIC" 혼재** — 나무위키 자료가 PIC 브랜드 전체(괌+사이판)를 다뤄 청크에 섞임
   - 해결 방향: Day 2 Agent 시스템 메시지에 "괌 가족여행 챗봇" 페르소나 명시 / Day 3 발표 폴리싱에서 시연 질문을 "괌 PIC 어때?"로 다듬기

2. **Q2 답변에서 시점·수치 환각** — 자료에 5월 강수량·시간대 정보 없는데 LLM이 일반 상식으로 채워넣음
   - 해결 방향: **Day 2 OpenWeatherMap 도구가 정확한 시점·수치 데이터 제공 → LLM이 환각할 동기 자체 사라짐 (자연 해결 가능성 높음)**
   - 도구 호출 후에도 환각 잔존하면 Day 3에 추가 프롬프트 강화 검토

3. **외교부 0404 보강 정책 유지** — 시점 정보 해결책은 도구이고, 외교부 0404는 안전·날씨 일반 정보 추가용으로만 (MVP 완성 후 검토)

4. **Q1 강화 후 회귀 검증 미실행** — 셀 5는 강화 *전* 프롬프트로 출력된 결과. Day 2 시작 전 노트북 재실행하면 셀 5도 강화 후 결과로 갱신됨. PIC 답변 풍부함이 떨어졌는지(트레이드오프) 그때 확인

5. **Q4 RAG 부분 검증 (선택) 미실행** — "PIC 추천해줘. 4박 5일 30만원이면 충분해?"의 RAG 부분은 안 했음. Day 2 시작 전 시간 있으면 추가

### 다음 작업
**Day 2 — Tools + Agent**: src/guam_chatbot/tools/*.py 4개 + agent.py + 04_tools_test.ipynb + 05_agent_test.ipynb (plan Section 9 Day 2 참조)